4. 군집분석 및 감성 분석 목차
  4.1 분석 목적
  4.2 분석 파이프라인
  4.3 데이터 개요 및 감성 점수 검증
      - 데이터 규모 (7,086개 문장, 13개 제품, 36개 속성)
      - VADER 감성 점수 타당성 확인 (극단 문장 사례)
  4.4 최적 군집 수(K) 결정
      - 엘보우 방법 (탐색범위, 자동 선택 로직)
      - 최종 K=12 결정
  4.5 군집별 주요 단어 및 주제 라벨링
      - 군집별 대표 단어표
      - 자동차/전자제품/호텔 3개 대주제로 구분됨
  4.6 군집별 평균 감성 분석
  4.7 Ridge 회귀를 이용한 군집-감성 관계 분석
      - 회귀 설명력(R², RMSE)과 해석
      - 예측이 아닌 해석이 목적임을 명시
  4.8 감성을 높이는 군집 TOP3
  4.9 감성을 낮추는 군집 TOP3
  4.10 군집 해석 시 주의사항
      - 유사 군집의 분리(cluster_7/8)
      - 혼합형 군집의 해석 제한(cluster_0)
  4.11 체르노프 페이스를 통한 종합 시각화
      - 매핑 규칙(얼굴크기/눈크기/눈썹/입/색)
      - 해석
  4.12 종합 해석

# 4. 군집분석 및 감성 분석

## 4.1 분석 목적

본 분석에서는 제품 리뷰 문장에 포함된 **주요 군집(주제)을 추출하고, 각 군집이 감성에 미치는 영향을 분석**하였다.

구체적인 분석 목적은 다음과 같다.

1. 전체 리뷰에서 어떤 군집(주제)이 존재하는지 파악
2. 군집별 평균 감성 점수의 차이 분석
3. 회귀계수를 기준으로 감성에 가장 큰 영향을 미치는 군집 파악
4. 동일한 군집이라도 제품군에 따라 감성 차이가 존재하는지 확인

---

## 4.2 분석 파이프라인

전체 분석 과정은 다음과 같이 구성하였다.

```text
51개 리뷰 파일
      ↓
① 데이터 로드 및 기본 노이즈 정제
      ↓
② VADER 감성 분석
   └─ 원문 기준 문장별 감성 점수 계산
      ↓
③ 군집분석용 텍스트 전처리
   └─ 감성어 및 불용어 제거
      ↓
④ TF-IDF 벡터화
      ↓
⑤ TruncatedSVD 차원축소
      ↓
⑥ 엘보우 방법으로 최적 군집 수(K) 탐색
      ↓
⑦ KMeans 군집분석
      ↓
⑧ 군집별 주요 단어 확인 (군집 중심 기준)
      ↓
⑨ 사람이 군집 라벨링
      ↓
⑩ Ridge 회귀
   └─ 군집 → 감성 점수 관계 분석
      ↓
⑪ 시각화 및 해석
   ├─ 군집별 평균 감성 점수
   ├─ 군집별 회귀계수
   ├─ 군집 × 제품 히트맵
   └─ 체르노프 페이스 (종합 시각화)
```

---

## 4.3 데이터 규모 및 감성 점수 확인

총 **7,086개 문장**을 대상으로 분석을 진행하였다. 데이터는 **13개 제품, 36개 속성(파일)**으로 구성되며, 51개 리뷰 파일이 정상적으로 로드되었다.

문장별 감성 점수는 VADER를 사용하여 원문을 기준으로 계산하였다.

VADER 감성 점수의 극단값을 확인한 결과, 실제 문장의 감성 방향과 일치하는 모습을 보였다.

* 가장 긍정적인 문장: **0.984**

  * "fun car to drive", "comfortable", "looking forward to a lot of great miles" 등 강한 긍정 표현 포함
* 가장 부정적인 문장: **-0.92**

  * "not fixable", "would not purchase another camry", "disappointed" 등 강한 부정 표현 포함

따라서 VADER의 감성 점수가 리뷰 문장의 전반적인 긍정·부정 방향을 비교적 적절하게 반영하고 있음을 확인하였다.

---

## 4.4 최적 군집 수(K) 결정

KMeans 군집분석을 위해 군집 수(K)를 **2~30** 범위에서 탐색하였다.

각 K에 대해 KMeans의 inertia(군집 내 분산)를 계산한 뒤, inertia 곡선에서 **첫 지점과 마지막 지점을 잇는 직선으로부터 가장 멀리 떨어진 지점**을 엘보우(꺾이는 지점)로 자동 판단하는 방식을 사용하였다.

그 결과 **K=12**가 최적 군집 수로 선택되었다.

이는 초기 분석에서 임의로 설정했던 K=12와 일치하지만, 이번에는 데이터 기반 탐색을 통해 결정된 값이라는 점에서 의미가 있다.

따라서 이후 분석에서는 **12개의 군집을 최종 군집 구조로 사용**하였다.

---

## 4.5 군집별 주요 단어 및 해석

TF-IDF 공간에서 각 군집의 중심(centroid)을 역변환하여 대표 단어를 확인하고, 주요 단어의 의미를 바탕으로 사람이 직접 군집에 대한 라벨을 부여하였다.

| 군집         | 대표 단어                                               | 주제 해석        |
| ---------- | --------------------------------------------------- | ------------ |
| cluster_0  | interior, battery, price, button, kindle, direction | 혼합/애매한 주제    |
| cluster_1  | seat, transmission, leather, driver seat, shift     | 자동차 시트/변속기   |
| cluster_2  | video, sound, camera, music, ipod, radio            | iPod 영상/음질   |
| cluster_3  | battery, life, battery life, hour, screen           | 배터리 지속시간     |
| cluster_4  | screen, size, keyboard, small, touch, read          | 화면 크기/터치     |
| cluster_5  | speed, speed limit, road, posted speed, highway     | GPS 속도제한 표시  |
| cluster_6  | room, small, bed, bathroom, hotel room, floor       | 호텔 객실 크기/침대  |
| cluster_7  | location, hotel, tube, wharf, fisherman wharf       | 호텔 위치        |
| cluster_8  | hotel location, staff, location staff, wharf        | 호텔 위치 + 응대   |
| cluster_9  | service, room service, food, breakfast              | 호텔 서비스/식사    |
| cluster_10 | mileage, gas, mpg, highway, drive, city             | 자동차 연비       |
| cluster_11 | staff, hotel staff, desk, front desk, concierge     | 호텔 프런트/직원 응대 |

전체적으로 군집은 크게 **자동차 관련 주제, 전자제품 관련 주제, 호텔 서비스 및 시설 관련 주제**로 구분되는 모습을 보였다.

---

## 4.6 군집별 평균 감성

군집별 평균 감성 점수를 비교함으로써 특정 주제가 언급될 때 리뷰의 전반적인 감성 방향이 어떻게 나타나는지 확인하였다.

특히 호텔 관련 군집에서는 **직원 응대와 위치 관련 군집이 상대적으로 높은 긍정 감성**을 보였으며, 자동차 관련 군집 중에서는 **시트 및 변속기 관련 군집의 감성이 가장 낮게 나타났다.**

다만 군집은 특정 단어의 등장 패턴을 기준으로 구성되기 때문에, 평균 감성만으로 해당 주제가 직접적으로 긍정 또는 부정의 원인이라고 단정해서는 안 된다.

---

## 4.7 Ridge 회귀를 이용한 군집과 감성의 관계 분석

군집이 감성 점수에 미치는 방향과 상대적인 영향력을 확인하기 위해 **Ridge 회귀**를 적용하였다. 군집 라벨은 원-핫 인코딩하여 회귀의 독립변수로 사용하였다.

회귀모형의 설명력은 다음과 같다.

* **R² = 0.15**
* **RMSE = 0.41**

R²가 0.15라는 것은 군집 정보만으로 문장 감성 점수 변동의 약 **15%를 설명할 수 있음**을 의미한다.

예측 모델 관점에서는 높은 설명력이라고 보기 어렵지만, 본 분석의 목적은 높은 예측 정확도를 확보하는 것이 아니라 **어떤 군집이 감성과 어떤 방향으로 연결되는지 해석하는 것**에 있다.

동일한 군집에서도 표현 방식에 따라 감성이 달라질 수 있다. 예를 들어 배터리라는 동일한 주제를 언급하더라도 "배터리가 오래간다"는 긍정적인 표현이고, "배터리가 금방 닳는다"는 부정적인 표현이다.

따라서 R²가 높지 않은 것은 자연스러운 결과이며, **군집 외에도 개별 단어와 문맥, 표현 방식 등이 감성에 중요한 영향을 미친다는 점**을 보여준다.

---

## 4.8 감성을 높이는 군집

Ridge 회귀계수가 양수인 군집 중 영향력이 큰 군집은 다음과 같다.

| 순위 | 군집         |  회귀계수 | 해석            |
| -- | ---------- | ----: | ------------- |
| 1  | cluster_8  | +0.21 | 호텔 위치 + 직원 응대 |
| 2  | cluster_11 | +0.19 | 호텔 프런트/컨시어지   |
| 3  | cluster_3  | +0.16 | 배터리 지속시간      |

특히 **cluster_8과 cluster_11이 높은 양의 회귀계수**를 보였다.

이는 호텔 리뷰에서 **직원 응대, 프런트, 컨시어지 등 사람이 직접 제공하는 서비스와 관련된 내용이 긍정적인 감성과 연결되는 경향**이 있음을 의미한다.

또한 cluster_3의 경우 배터리 지속시간과 관련된 군집으로, 배터리 사용시간에 대한 긍정적인 평가가 감성 점수를 높이는 방향으로 나타난 것으로 해석할 수 있다.

---

## 4.9 감성을 낮추는 군집

반대로 음의 회귀계수를 갖는 군집 중 영향력이 큰 군집은 다음과 같다.

| 순위 | 군집         |  회귀계수 | 해석         |
| -- | ---------- | ----: | ---------- |
| 1  | cluster_1  | -0.41 | 자동차 시트/변속기 |
| 2  | cluster_0  | -0.08 | 혼합/애매한 주제  |
| 3  | cluster_10 | -0.07 | 자동차 연비     |

가장 눈에 띄는 군집은 **cluster_1**이다.

cluster_1의 회귀계수는 **-0.41**로 다른 군집에 비해 절대값이 크게 나타났다.

해당 군집의 주요 단어는 `seat`, `transmission`, `leather`, `driver seat`, `shift` 등이며, 자동차의 **좌석 편의성과 변속감**에 대한 내용이 중심이다.

실제 원문에서도 `uncomfortable`, `kills my back`, VCM 변속 이질감 등 불편함이나 불만을 나타내는 내용이 확인되었다.

따라서 **자동차 시트의 편안함과 변속 품질이 해당 제품군에서 중요한 부정적 평가 요인으로 나타났다**고 해석할 수 있다.

---

## 4.10 군집 해석 시 주의사항

### ① cluster_7과 cluster_8의 주제 중복

cluster_7과 cluster_8은 모두 `location`, `wharf` 등의 단어를 포함하고 있어 사실상 **호텔 위치와 관련된 유사한 군집**으로 볼 수 있다.

특히 `fisherman wharf`, `tube`, `gloucester`와 같은 제품별 고유명사가 포함되면서 하나의 위치 관련 주제가 여러 군집으로 분리된 것으로 판단된다.

따라서 해석 단계에서는 두 군집을 별개의 완전히 독립적인 주제로 보기보다는 **'호텔 위치'라는 상위 주제 안에서 세분화된 군집**으로 보는 것이 적절하다.

### ② cluster_0의 해석 제한

cluster_0은 `interior`, `battery`, `kindle`, `direction` 등 서로 다른 제품과 관련된 단어가 혼합되어 있다.

따라서 명확한 주제라고 보기 어려운 **혼합형 군집**에 해당한다.

이 때문에 cluster_0의 회귀계수인 **-0.08에 지나치게 큰 의미를 부여하는 것은 적절하지 않다.**

---

## 4.11 체르노프 페이스를 통한 종합 시각화

앞서 확인한 군집별 통계치(문장 수, 평균 감성, 감성 표준편차, 회귀계수)를 하나의 그림으로 종합하기 위해 **체르노프 페이스(Chernoff Face)**를 이용한 시각화를 추가로 수행하였다.

체르노프 페이스는 다변량 데이터를 사람의 얼굴 특징에 매핑하여 여러 지표를 동시에 직관적으로 비교할 수 있도록 하는 시각화 기법이다. 각 얼굴 요소는 다음과 같이 매핑하였다.

| 얼굴 요소 | 매핑 데이터 | 해석 |
| --- | --- | --- |
| 얼굴 크기 | 군집 내 문장 수 | 클수록 리뷰에서 많이 언급된 주제 |
| 눈 크기 | 감성 점수 표준편차 | 클수록 해당 주제에 대한 의견이 긍정·부정으로 혼재 |
| 눈썹 각도 | 회귀계수 | 위로 갈수록 감성을 강하게 끌어올리는 군집, 아래로 갈수록 강하게 끌어내리는 군집 |
| 입 모양 | 평균 감성 점수 | 웃는 정도가 클수록 대체로 긍정적으로 언급되는 주제 |
| 얼굴 색 | 회귀계수 부호 | 초록(긍정 영향) / 빨강(부정 영향) / 회색(중립) |

시각화 결과, **cluster_1(자동차 시트/변속기)**은 눈썹이 아래로 처지고 입이 찡그려진 붉은색 얼굴로 나타나 가장 강한 부정적 군집임을 한눈에 확인할 수 있었다. 반대로 **cluster_8(호텔 위치+응대)**은 눈썹이 위로 올라가고 활짝 웃는 초록색 얼굴로 나타나, 회귀계수·평균 감성 분석 결과와 일관된 패턴을 시각적으로도 확인하였다.

이처럼 체르노프 페이스는 여러 개의 수치 지표를 표나 개별 그래프로 나열하지 않고도, 군집 간 차이를 직관적으로 비교할 수 있게 해준다는 점에서 보조적인 해석 도구로 활용하였다.

---

## 4.12 종합 해석

이번 군집분석을 통해 리뷰 데이터에서 **자동차, 전자제품, 호텔 서비스 및 시설**과 관련된 다양한 주제가 존재함을 확인하였다.

특히 Ridge 회귀 분석 결과, 군집별 감성 영향에는 뚜렷한 차이가 나타났다.

* **호텔 직원 응대 및 프런트 관련 군집**은 긍정적인 감성과 강하게 연결
* **배터리 지속시간** 역시 긍정적인 방향으로 연결
* **자동차 시트 및 변속기 관련 군집**은 가장 강한 부정적 영향
* **연비 관련 군집**은 상대적으로 약한 부정적 영향

다만 군집만으로 감성의 약 15% 정도를 설명할 수 있었기 때문에, **군집 자체가 감성을 결정한다기보다 특정 주제가 어떤 맥락과 표현으로 언급되는지가 감성에 더 큰 영향을 미친다**고 볼 수 있다.

또한 군집분석 과정에서 유사한 군집이 분리되거나 여러 제품의 단어가 하나의 군집에 혼합되는 현상이 확인되었다. 따라서 군집분석 결과는 단순한 군집 결과로 해석하기보다 **대표 단어, 실제 리뷰 문장, 제품군별 분포, 체르노프 페이스와 같은 종합 시각화를 함께 확인하는 방식으로 해석하는 것이 적절하다.**